# FRQI / NEQR Encoding — Starter Notebook

Goal: implement pixel-to-qubit encoding for both FRQI and NEQR on a small grayscale patch, compare qubit count and reconstruction accuracy.

Start with a tiny patch (e.g. 2x2 or 4x4) to keep qubit count manageable while validating correctness.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit import transpile
import numpy as np
import matplotlib.pyplot as plt

## 1. Load a tiny sample patch

Use a real document image crop once data is available; start with a synthetic patch to validate the encoding logic first.

In [ ]:
# Synthetic 2x2 grayscale patch, values 0-255
patch = np.array([
    [0, 255],
    [128, 64]
])
print(patch)

## 2. FRQI Encoding

One color qubit (rotation angle = pixel value), position qubits encode pixel index.
For a 2x2 patch (4 pixels), need 2 position qubits + 1 color qubit = 3 qubits total.

In [ ]:
def build_frqi_circuit(patch):
    flat = patch.flatten()
    n_pixels = len(flat)
    n_pos_qubits = int(np.ceil(np.log2(n_pixels)))

    # angles: theta_i = (pi/2) * pixel_value / 255
    angles = (np.pi / 2) * (flat / 255.0)

    pos = QuantumRegister(n_pos_qubits, 'pos')
    color = QuantumRegister(1, 'color')
    qc = QuantumCircuit(pos, color)

    # uniform superposition over positions
    qc.h(pos)

    # TODO: apply controlled-Ry(2*theta_i) on color qubit,
    # controlled on position register == i, for each pixel i.
    # This is the core FRQI encoding step — fill in using
    # multi-controlled rotation gates (mcry or decomposition).

    return qc, n_pos_qubits, angles

qc, n_pos_qubits, angles = build_frqi_circuit(patch)
print(f"Position qubits: {n_pos_qubits}, total qubits: {n_pos_qubits + 1}")
print(f"Angles: {angles}")
qc.draw('mpl')

## 3. NEQR Encoding

Exact binary grayscale value per pixel (8 qubits for 0-255), entangled with position qubits.
For a 2x2 patch: 2 position qubits + 8 color qubits = 10 qubits total.

In [ ]:
def build_neqr_circuit(patch, bit_depth=8):
    flat = patch.flatten()
    n_pixels = len(flat)
    n_pos_qubits = int(np.ceil(np.log2(n_pixels)))

    pos = QuantumRegister(n_pos_qubits, 'pos')
    color = QuantumRegister(bit_depth, 'color')
    qc = QuantumCircuit(pos, color)

    qc.h(pos)

    # TODO: for each pixel i, apply multi-controlled X gates
    # (controlled on position == i) to set the color register
    # to the exact binary value of that pixel.

    return qc, n_pos_qubits, bit_depth

qc_neqr, n_pos_qubits, bit_depth = build_neqr_circuit(patch)
print(f"Position qubits: {n_pos_qubits}, color qubits: {bit_depth}, total: {n_pos_qubits + bit_depth}")
qc_neqr.draw('mpl')

## 4. Comparison

Once both circuits are complete: compare qubit counts, run on AerSimulator, and check reconstruction accuracy
(FRQI: repeated measurement + angle estimation; NEQR: direct exact readout).

In [ ]:
# TODO: comparison table
# - qubit count: FRQI vs NEQR
# - reconstruction accuracy on this patch
# - measurement shots needed for stable FRQI angle estimate